In [1]:
!pip install -q llama_stack_client

In [2]:
from llama_stack_client import LlamaStackClient

LLAMA_ENDPOINT = "http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com"

client = LlamaStackClient(base_url=LLAMA_ENDPOINT)

In [3]:
models = client.models.list()

INFO:httpx:HTTP Request: GET http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/models "HTTP/1.1 200 OK"


In [4]:
# values from llamastack api http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/models

model_id = "vllm-inference/llama3"

embedding_model = "vllm-embedding/Qwen/Qwen3-Embedding-0.6B"
embedding_model_id = "vllm-embedding/Qwen/Qwen3-Embedding-0.6B"
embedding_dimension = 1024

In [5]:
# Optional - Test embeddings 
print("Testing embeddings...")

texts = [
    "OpenShift AI is a machine learning platform",
    "Llama Stack provides RAG capabilities",
    "Kubernetes orchestrates containerized workloads"
]

response = client.embeddings.create(
    model="vllm-embedding/Qwen/Qwen3-Embedding-0.6B",  # ← full ID
    input=texts
)

print(f" Got {len(response.data)} embeddings")
for i, (text, embedding) in enumerate(zip(texts, response.data)):
    print(f"\n  Text {i+1}: '{text[:50]}'")
    print(f"  Vector dims: {len(embedding.embedding)}")
    print(f"  First 5 values: {[round(v, 4) for v in embedding.embedding[:5]]}")

Testing embeddings...


INFO:httpx:HTTP Request: POST http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/embeddings "HTTP/1.1 200 OK"


 Got 3 embeddings

  Text 1: 'OpenShift AI is a machine learning platform'
  Vector dims: 1024
  First 5 values: [0.008, -0.0127, -0.0095, -0.094, -0.0121]

  Text 2: 'Llama Stack provides RAG capabilities'
  Vector dims: 1024
  First 5 values: [0.019, 0.0185, -0.0104, -0.1052, 0.001]

  Text 3: 'Kubernetes orchestrates containerized workloads'
  Vector dims: 1024
  First 5 values: [0.0342, -0.0419, -0.0111, -0.0667, 0.0398]


In [6]:
# Optional - checking function vector_stores.create signature
import inspect
print(inspect.signature(client.vector_stores.create))

# Inspect all vector_store related methods
print("vector_stores methods:")
for attr in dir(client.vector_stores):
    if not attr.startswith("_"):
        print(f"  {attr}")

(*, chunking_strategy: 'Optional[vector_store_create_params.ChunkingStrategy] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, expires_after: 'Optional[Dict[str, object]] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, file_ids: 'Optional[SequenceNotStr[str]] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, metadata: 'Optional[Dict[str, object]] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, name: 'Optional[str] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, extra_headers: 'Headers | None' = None, extra_query: 'Query | None' = None, extra_body: 'Body | None' = None, timeout: 'float | httpx.Timeout | None | NotGiven' = NOT_GIVEN) -> 'VectorStore'
vector_stores methods:
  create
  delete
  file_batches
  files
  list
  retrieve
  search
  update
  with_raw_response
  with_streaming_response


In [7]:
# Cell - Register a vector DB
vector_store = client.vector_stores.create(
    name="my_inline_milvus",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "milvus",
    },
)
vector_store_id = vector_store.id

INFO:httpx:HTTP Request: POST http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/vector_stores "HTTP/1.1 200 OK"


In [8]:
# Optional - checking fuction signature
import inspect
print(inspect.signature(client.vector_stores.files.create))

(vector_store_id: 'str', *, file_id: 'str', attributes: 'Optional[Dict[str, object]] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, chunking_strategy: 'Optional[file_create_params.ChunkingStrategy] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, extra_headers: 'Headers | None' = None, extra_query: 'Query | None' = None, extra_body: 'Body | None' = None, timeout: 'float | httpx.Timeout | None | NotGiven' = NOT_GIVEN) -> 'VectorStoreFile'


In [9]:
# Ingest documents
import requests

pdf_url = "https://www.federalreserve.gov/aboutthefed/files/quarterly-report-20250822.pdf"
filename = "quarterly-report-20250822.pdf"

response = requests.get(pdf_url)
response.raise_for_status()

with open(filename, "wb") as f:
    f.write(response.content)

with open(filename, "rb") as f:
    file_info = client.files.create(
        file=(filename, f),
        purpose="assistants",
    )

vector_store_file = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 800,
            "chunk_overlap_tokens": 400,
        },
    },
)

print(vector_store_file)

INFO:httpx:HTTP Request: POST http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/vector_stores/vs_b50b6de6-ef5c-47ff-9ae0-55f3c2f4a06b/files "HTTP/1.1 200 OK"


VectorStoreFile(id='file-aa9813809191496b855f646f7e9ba609', chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'), created_at=1773333763, status='completed', vector_store_id='vs_b50b6de6-ef5c-47ff-9ae0-55f3c2f4a06b', attributes={}, last_error=None, object='vector_store.file', usage_bytes=0)


In [10]:
# instructions
system_instructions = """You are a precise and reliable AI assistant.
Use retrieved context when it is available.
If nothing relevant is found, say so clearly."""

query="for the quarter-ended June 30, 2025 -summarize the combined financial position and results of operations of the 12 Federal Reserve Banks. Short and neat"

In [11]:
# Optional - checking fuction signature
import inspect
print(inspect.signature(client.responses.create))

(*, input: 'Union[str, Iterable[response_create_params.InputListOpenAIResponseMessageUnionOpenAIResponseInputFunctionToolCallOutput]]', model: 'str', conversation: 'Optional[str] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, include: "Optional[List[Literal['web_search_call.action.sources', 'code_interpreter_call.outputs', 'computer_call_output.output.image_url', 'file_search_call.results', 'message.input_image.image_url', 'message.output_text.logprobs', 'reasoning.encrypted_content']]] | Omit" = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, instructions: 'Optional[str] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, max_infer_iters: 'Optional[int] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, max_tool_calls: 'Optional[int] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, metadata: 'Optional[Dict[str, str]] | Omit' = <llama_stack_client.Omit object at 0x7f5425ccc5c0>, parallel_tool_calls: 'Optional[bool] | Omit' = <llama_s

In [12]:
# Query without using a vector store
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
)

print(response.output_text)

INFO:httpx:HTTP Request: POST http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/responses "HTTP/1.1 200 OK"


I don't have access to real-time or future financial data. However, I can provide a general summary based on publicly available information up to 2023.

As of 2023, the combined financial position and results of operations of the 12 Federal Reserve Banks were:

**Combined Financial Position:**

- Total assets: Over $5.1 trillion
- Total liabilities: Over $2.4 trillion
- Net worth: Over $2.7 trillion

**Combined Results of Operations:**

- Net income: Over $5.6 billion
- Return on assets (ROA): Approximately 0.11%
- Return on equity (ROE): Approximately 0.25%

Please note that these figures are not specific to the quarter-ended June 30, 2025, and may not reflect the current financial situation of the Federal Reserve Banks. For the most accurate and up-to-date information, I recommend checking the Federal Reserve's official website or contacting their public information office.


In [13]:
#Query by using the Responses API with file search
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
            "max_num_results": 3
        }
    ],
)

print(response.output_text)

INFO:httpx:HTTP Request: POST http://lsd-llama-milvus-inline-service-demo-llm.apps.aba-sno.test.com/v1/responses "HTTP/1.1 200 OK"


"Based on the Federal Reserve Board's combined quarterly financial report for the quarter-ended June 30, 2025, the combined financial position and results of operations of the 12 Federal Reserve Banks are as follows:

Total assets: $6,905,127 million
Total liabilities and capital: $6,905,127 million

The report contains eight explanatory notes that provide supplemental financial information for line items in the combined quarterly statements.

For more information about Federal Reserve Board financial statements and reporting, visit our website at https://www.federalreserve.gov/aboutthefed/fed-financial-statements.htm. For more information about how the Federal Reserve Board supervises Federal Reserve Bank operations, see the \"Payment System and Reserve Bank Oversight\" section of our latest Annual Report (https://www.federalreserve.gov/publications/annual-report.htm)."

{"name": "knowledge_search", "parameters": {"query": "Combined financial position and results of operations of the 